# EcoData · Sesión 12 (U2) · Dimensión CLIMA (Alahin)
**Equipo EcoData** — Lanzeloth Vargas Marichi · Alahin Reyme Ticona Veliz · Big Data, UPeU

Flujo de la dimensión clima: `Open-Meteo API → Kafka (atmos-clima) → Spark Structured Streaming + modelo GBT (MLlib) → Pushgateway → Prometheus → Grafana`

| Sección | Qué demuestra |
|---|---|
| 0 | Preparación |
| 1 | Conexión con la API (clima) |
| 2 | Verificar y contar eventos en Kafka (`atmos-clima`) |
| 3 | Modelo de temperatura (metadatos) |
| 4 | Jobs de streaming (solo estado) |
| 5 | Métricas de clima en Prometheus / Pushgateway |

> Ejecuta las celdas en orden con Shift+Enter. **No** uses "Run All". Requiere Kafka, Pushgateway y Prometheus levantados (Docker). Si el kernel se reinicia, empieza siempre por la sección 0.

## 0. Preparación

In [ ]:
import sys, json, time, subprocess, urllib.request, urllib.parse
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import clima_comun as clima
import openmeteo_clima_a_kafka as pub_clima

BOOTSTRAP = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS_CANDIDATOS = ["http://prometheus:9090", "http://lambda26-prometheus:9090"]
print("Scripts del proyecto cargados desde /tmp/s12/scripts")

from kafka import KafkaConsumer, TopicPartition
c = KafkaConsumer(bootstrap_servers=BOOTSTRAP)
print("Kafka OK. Topics:", sorted(t for t in c.topics() if t.startswith("atmos")))
c.close()

## 1. Conexión con la API (Open-Meteo, clima)
Open-Meteo entrega datos **horarios y modelados** (reanálisis/pronóstico), no sensores en vivo. La función `desde_api()` del productor de clima consulta la API y devuelve las horas ya ocurridas.

In [ ]:
serie_clima = pub_clima.desde_api()
df_clima = pd.DataFrame(serie_clima)
print(f"{len(df_clima)} horas recibidas de la API")
df_clima.tail(8)

## 2. Verificar y contar eventos en Kafka (`atmos-clima`)
El campo `origen` dice de dónde vino cada dato: `openmeteo-api` (API), `openmeteo-replay` (histórico reproducido) o `demo-tardio` (dato tardío de prueba).

In [ ]:
c = KafkaConsumer("atmos-clima", bootstrap_servers=BOOTSTRAP, auto_offset_reset="earliest",
                  enable_auto_commit=False, consumer_timeout_ms=8000)
todos = [json.loads(m.value.decode("utf-8")) for m in c]
c.close()
df_k = pd.DataFrame(todos)
print(len(df_k), "eventos leídos en atmos-clima")
if "origen" in df_k:
    display(df_k["origen"].value_counts())
if "timestamp" in df_k:
    print("Horas distintas:", df_k["timestamp"].nunique())
    df_k["hora_UTC"] = pd.to_datetime(df_k["timestamp"], unit="ms")
df_k.tail(10)

## 3. Modelo de temperatura
GBT de MLlib que predice el **cambio** de temperatura a +1 h; se compara contra la persistencia y la persistencia diaria.

In [ ]:
import os
ruta = "/tmp/s12/modelo_temp/meta.json"
if os.path.exists(ruta):
    meta = json.load(open(ruta))
    print("Meta del modelo de clima:")
    for k, v in meta.items():
        print(f"  {k}: {v}")
else:
    print("No se encontró", ruta, "- revisa que el modelo esté en /tmp/s12/modelo_temp")

## 4. Jobs de streaming (solo estado)
Spark lee de Kafka, calcula ventanas de 6 h con watermark de 2 h, infiere y empuja métricas al Pushgateway. **Un solo job por tipo**: no lances otro si ya están corriendo.

In [ ]:
def jobs_activos():
    out = subprocess.run("ps aux | grep s12_streaming | grep -v grep", shell=True, capture_output=True, text=True).stdout
    return [l for l in out.splitlines() if "spark-submit" in l or "SparkSubmit" in l]
print("Procesos spark activos:", len(jobs_activos()))
for l in jobs_activos():
    print(" ", l[:140])

## 5. Métricas de clima en Prometheus (lo mismo que ve Grafana)

In [ ]:
def prom(q):
    for base in PROMETHEUS_CANDIDATOS:
        try:
            u = f"{base}/api/v1/query?" + urllib.parse.urlencode({"query": q})
            return json.load(urllib.request.urlopen(u, timeout=4))["data"]["result"]
        except Exception as e:
            err = e
    raise err

consultas = [
    ("Frescura clima (s)", 'time()-push_time_seconds{job="lambda26_clima",instance="inferencia"}'),
    ("Temperatura última (°C)", "lambda26_clima_temp_ultima"),
    ("Predicción próxima hora (°C)", "lambda26_clima_pred_temp_proxima_hora"),
    ("MAE modelo (°C)", "lambda26_clima_mae_modelo"),
    ("MAE persistencia (°C)", "lambda26_clima_mae_persistencia"),
    ("Mejora vs persistencia (%)", "lambda26_clima_mejora_pct"),
    ("Horas evaluadas", "lambda26_clima_n_evaluadas"),
    ("Heladas detectadas", "lambda26_clima_heladas_total"),
    ("Filas descartadas (watermark)", "lambda26_clima_filas_descartadas_watermark_total"),
]
for nombre, q in consultas:
    try:
        r = prom(q)
        print(f"{nombre:34s}", [round(float(x["value"][1]), 3) for x in r] or "sin datos")
    except Exception as e:
        print(nombre, "->", e)

Alternativa directa: leer el Pushgateway.

In [ ]:
txt = urllib.request.urlopen(PUSHGATEWAY + "/metrics", timeout=4).read().decode()
print("\n".join(l for l in txt.splitlines() if l.startswith("lambda26_clima_") and not l.startswith("#")))

**Dashboard Grafana:** http://localhost:43000/d/lambda26-s12-ecodata (admin / admin) · **Prometheus:** http://localhost:49090 · **Kafka UI:** http://localhost:48085